In [ ]:
DRIVE_DIR   = "/content/drive/MyDrive/VTUAV"
DATA_ROOT   = "/content/data/VTUAV_lt_ir"
POOL_ROOT   = "/content/pool"
POOL        = "vtuav_lt_thermal"
MODALITY    = "ir"
EXTRACT_MODE = "tracked_ir"
MIRROR_DIR  = "/content/drive/MyDrive/edgetam-pool/vtuav_lt_thermal"
ARCHIVES    = ["train_LT_001.zip", "train_LT_002.zip",
               "train_LT_003.zip", "train_LT_004.zip"]
TEACHER     = "facebook/sam3"
DTYPE       = "bfloat16"
ZOOM        = 4.0
MIN_SIZE    = 128
MIN_LUMA    = 0.0
BATCH       = 0
FRAME_GROUP = 0
READERS     = 0
READ_AHEAD  = 0
UNZIP_WORKERS = 16
MAX_BOXES   = None
LIMIT       = None
BOX_IOU     = 0.6
REPO_URL    = "https://github.com/yigitkayabagci/sam-dedection.git"
BRANCH      = "claude/thermal-stage-b-training-43ktcl"
REPO_DIR    = "/content/sam-dedection"
NOTEBOOK = "notebooks/25_vtuav_lt_thermal_pool.ipynb".split("/")[-1]
STAMP    = "73ebd5090d"

import json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

if Path(REPO_DIR).exists():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1",
                    "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"],
                   check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import importlib
for _stale in [_m for _m in list(sys.modules)
               if _m.split(".")[0] in ("src", "tools")]:
    del sys.modules[_stale]
importlib.invalidate_caches()
print("repo at", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short",
                                 "HEAD"], capture_output=True,
                                text=True).stdout.strip())

_missing = []
try:
    import transformers as _transformers
    if int(_transformers.__version__.split(".")[0]) < 5:
        _missing.append("transformers>=5.0.0")
except Exception:
    _missing.append("transformers>=5.0.0")
for _name in ("accelerate", "huggingface_hub", "tqdm"):
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)
if _missing:
    print("installing", _missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    *_missing], check=False)
    raise SystemExit(
        "installed " + ", ".join(_missing) + " into a running kernel. "
        "Runtime > Restart session, then run this cell again. pip replaced "
        "files this kernel has already imported, and the half-old half-new "
        "import that follows fails somewhere unrelated -- PIL, torchvision -- "
        "rather than here.")

try:
    from google.colab import drive as _drive
    _drive.mount("/content/drive")
    try:
        next(Path("/content/drive/MyDrive").iterdir(), None)
    except OSError as _stale_mount:
        print("stale Drive mount:", _stale_mount, "-- remounting")
        _drive.mount("/content/drive", force_remount=True)
except Exception as _mount_error:
    print("no Colab Drive mount:", _mount_error)

try:
    from google.colab import userdata as _userdata
    _token = _userdata.get("HF_TOKEN")
    if _token:
        os.environ["HF_TOKEN"] = _token
except Exception as _token_error:
    print("no HF_TOKEN secret:", _token_error)

if os.environ.get("HF_TOKEN"):
    try:
        from huggingface_hub import login as _login
        _login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    except Exception as _login_error:
        print("hf login skipped:", _login_error)

import torch, cv2
from src.training.pool import read_workers

cv2.setNumThreads(1)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

_device = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
VRAM = round(_device.total_memory / 2 ** 30, 1) if _device else 0.0
CORES = os.cpu_count() or 1
if BATCH <= 0:
    BATCH = max(4, int(VRAM * 0.8)) if VRAM else 4
if FRAME_GROUP <= 0:
    FRAME_GROUP = BATCH
READERS = read_workers(READERS)
if READ_AHEAD <= 0:
    READ_AHEAD = max(2 * FRAME_GROUP, READERS)

def progress(stream, total, desc):
    from tqdm.auto import tqdm
    return tqdm(stream, total=total, desc=desc)

_stamps = Path(REPO_DIR) / "notebooks" / ".stamps.json"
_want = json.loads(_stamps.read_text()).get(NOTEBOOK) if _stamps.is_file() else None
print(NOTEBOOK, STAMP, "| repo:", _want,
      "| OK" if _want == STAMP else "| STALE, re-open from the repo")
print(_device.name if _device else "no GPU", VRAM, "GiB",
      "| BATCH", BATCH, "| FRAME_GROUP", FRAME_GROUP, "| modality", MODALITY)
print(CORES, "cores | READERS", READERS, "| READ_AHEAD", READ_AHEAD,
      "|", round(READ_AHEAD * 1920 * 1080 * 3 / 2 ** 30, 2), "GiB of frames "
      "in flight | UNZIP_WORKERS", UNZIP_WORKERS)


In [ ]:
import math
from src.training.boxes import annotated_stride

def absent_row(text):
    cells = text.replace(",", " ").split()
    if len(cells) < 4:
        return True
    try:
        x, y, w, h = (float(v) for v in cells[:4])
    except ValueError:
        return True
    return not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0

MISSING, PROBE = [], []
for _archive in ARCHIVES:
    _source = Path(DRIVE_DIR) / _archive
    if not _source.is_file():
        MISSING.append(_archive)
        continue
    _rows, _present, _gone = {}, {}, {}
    with zipfile.ZipFile(_source) as _zip:
        for _name in _zip.namelist():
            _parts = _name.split("/")
            if len(_parts) == 2 and _parts[1] == f"{MODALITY}.txt":
                _lines = [_l for _l in _zip.read(_name).decode(
                    "utf-8", "replace").splitlines() if _l.strip()]
                _rows[_parts[0]] = len(_lines)
                _gone[_parts[0]] = sum(1 for _l in _lines if absent_row(_l))
            elif len(_parts) == 3 and _parts[1] == MODALITY:
                if Path(_parts[-1]).stem.isdigit():
                    _present[_parts[0]] = _present.get(_parts[0], 0) + 1
    _strides, _unreadable = {}, []
    for _sequence, _lines in sorted(_rows.items()):
        try:
            _step = annotated_stride(_present.get(_sequence, 0), _lines)
        except ValueError as _mismatch:
            _unreadable.append(f"{_sequence}: {_mismatch}")
            continue
        _strides[_step] = _strides.get(_step, 0) + 1
    PROBE.append({"archive": _archive, "sequences": len(_rows),
                  "frames": sum(_present.values()), "rows": sum(_rows.values()),
                  "absent": sum(_gone.values()), "strides": _strides,
                  "unreadable": _unreadable,
                  "GiB": round(_source.stat().st_size / 2 ** 30, 1)})

print(f"{'archive':<22}{'seq':>5}{'frames':>10}{'rows':>8}{'absent':>9}"
      f"{'GiB':>7}  strides")
for _row in PROBE:
    _share = f"{_row['absent'] / max(_row['rows'], 1):.1%}"
    print(f"{_row['archive']:<22}{_row['sequences']:>5}{_row['frames']:>10}"
          f"{_row['rows']:>8}{_share:>9}{_row['GiB']:>7}  {_row['strides']}")
    for _line in _row["unreadable"]:
        print("   !! dropped --", _line)
if MISSING:
    print("\nnot in", DRIVE_DIR, "--", MISSING)
KEEPS = sum(_r["rows"] for _r in PROBE)
PROMPTS = sum(_r["rows"] - _r["absent"] for _r in PROBE)
print(f"\n{KEEPS} annotated {MODALITY} frame(s) to extract, {PROMPTS} of them "
      f"with a target to prompt; "
      f"{round(sum(_r['GiB'] for _r in PROBE), 1)} GiB read from Drive, "
      f"about {round(sum(_r['GiB'] for _r in PROBE) / 20, 1)} GiB on disk.")
assert PROBE, f"no archive of {ARCHIVES} is under {DRIVE_DIR} -- set DRIVE_DIR"
if any(set(_r["strides"]) - {10} or _r["unreadable"] for _r in PROBE):
    print("\n!! a stride other than 10, or a sequence whose counts imply "
          "none. The extractor and the indexer derive it the same way this "
          "cell does, so the harvest follows what is printed above -- but "
          "read it before spending the GPU-hours.")


In [ ]:
from tools.fetch_datasets import extract

Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
_done = Path(DATA_ROOT) / "_staged"
_done.mkdir(parents=True, exist_ok=True)
for _archive in ARCHIVES:
    _source = Path(DRIVE_DIR) / _archive
    _marker = _done / _archive
    if _marker.exists():
        print("already staged:", _archive)
        continue
    if not _source.is_file():
        raise SystemExit(f"{_source} is not there -- set DRIVE_DIR in cell 1")
    print("staging", _archive,
          round(_source.stat().st_size / 2 ** 30, 2), "GiB from Drive")
    extract(_source, Path(DATA_ROOT), frames=EXTRACT_MODE,
            workers=UNZIP_WORKERS)
    _marker.write_text("ok")

from src.training import boxes as B

FRAMES = B.vtuav_frames(DATA_ROOT, modality=MODALITY)
SEQUENCES = sorted({f.key.split("/")[0] for f in FRAMES})
print(len(FRAMES), "labelled frames /", len(SEQUENCES), "sequences")
print(B.summarise_frames(FRAMES, POOL))
print(round(sum(p.stat().st_size for p in Path(DATA_ROOT).rglob("*")
                if p.is_file()) / 2 ** 30, 2), "GiB on disk")


In [ ]:
import numpy as np, cv2
import matplotlib.pyplot as plt
from src.training.labels import Gates, build_image_teacher
from src.training.pool import label_boxes

TEACHER_MODEL = build_image_teacher(TEACHER, dtype=DTYPE)
GATES = Gates(box_iou=BOX_IOU)
print("teacher:", TEACHER)

_shown = [FRAMES[i] for i in
          np.linspace(0, len(FRAMES) - 1, 4).astype(int).tolist()]
_figure, _axes = plt.subplots(2, 2, figsize=(14, 9))
for _panel, _frame in zip(_axes.ravel(), _shown):
    _image = cv2.cvtColor(cv2.imread(str(_frame.image)), cv2.COLOR_BGR2RGB)
    _boxes, _keep = _frame.resolved(_image.shape[:2])
    _masks, _rows = label_boxes(_image, _boxes[_keep], TEACHER_MODEL,
                                gates=GATES, zoom=ZOOM, min_size=MIN_SIZE,
                                batch_size=BATCH)
    _x0, _y0, _x1, _y1 = (int(v) for v in _boxes[0])
    for _mask in _masks.values():
        _image[_mask] = (0.4 * _image[_mask]
                         + 0.6 * np.array([255, 40, 40])).astype(np.uint8)
    _pad = 200
    _panel.imshow(_image[max(_y0 - _pad, 0):_y1 + _pad,
                         max(_x0 - _pad, 0):_x1 + _pad])
    _panel.set_title(f"{_frame.key} {_frame.classes[0]} "
                     f"{len(_masks)}/{len(_rows)}")
    _panel.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
from src.training.pool import (label_pool, pool_report, summarise_luma,
                               summarise_pool, write_index)

def harvest(frames, dataset):
    global BATCH, FRAME_GROUP
    while True:
        try:
            return label_pool(frames, TEACHER_MODEL, POOL_ROOT, dataset=dataset,
                              prompt="self", gates=GATES, zoom=ZOOM,
                              min_size=MIN_SIZE, batch_size=BATCH,
                              frame_group=FRAME_GROUP, limit=LIMIT,
                              max_boxes=MAX_BOXES, min_luma=MIN_LUMA,
                              readers=READERS, read_ahead=READ_AHEAD,
                              progress=progress)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            BATCH = max(1, BATCH // 2)
            FRAME_GROUP = max(1, FRAME_GROUP // 2)
            print("out of memory, retrying at BATCH", BATCH,
                  "FRAME_GROUP", FRAME_GROUP)

REPORT = harvest(FRAMES, POOL)
print(json.dumps(REPORT, indent=1))
write_index(POOL_ROOT)
print(summarise_pool(pool_report(POOL_ROOT)))
print()
print(summarise_luma(POOL_ROOT))


In [ ]:
Path(MIRROR_DIR).mkdir(parents=True, exist_ok=True)
_target = Path(MIRROR_DIR) / f"{POOL}.zip"
_files = [p for p in sorted((Path(POOL_ROOT) / POOL).rglob("*")) if p.is_file()]
with zipfile.ZipFile(_target, "w", zipfile.ZIP_STORED, allowZip64=True) as _zip:
    for _file in _files:
        _zip.write(_file, _file.relative_to(Path(POOL_ROOT)))
print(_target, len(_files), "files",
      round(_target.stat().st_size / 2 ** 20, 1), "MiB")

_index = Path(POOL_ROOT) / "pool_index.jsonl"
if _index.is_file():
    shutil.copy(_index, Path(MIRROR_DIR) / _index.name)
print("teacher:", TEACHER, "| modality:", MODALITY,
      "| accepted:", REPORT["accepted"], "of", REPORT["attempted"])
